# NER Performance Evaluation Pipeline

In [38]:
'''
Function: ner_few_shot
Calls in: annotate (NER Performance Evaluation Pipeline)
Description:
 - Performs few-shot NER using an OpenAI model.
 - Takes instruction, example, paragraph, model name, and temperature as parameters.
 - Sends instruction, example, and paragraph to the OpenAI Chat API.
 - Receives and cleans the model's response.
 - Returns the cleaned response as JSON.
'''

from openai import OpenAI
import ast
import traceback

def ner_few_shot(instruction, example, paragraph, model, temperature):
    client = OpenAI()
    user_query = '''
        EXAMPLES:
        {example}
        
        TEXT:
        {paragraph}
    '''
    
    try:
        response = client.chat.completions.create(
          model = model,
          temperature = temperature,
          messages = [
            {'role': 'system', 'content': instruction},
            {'role': 'user', 'content': user_query.format(example=example, paragraph=paragraph)}
          ]
        )

        content = response.choices[0].message.content
        cleaned_content = content.strip().splitlines()
        cleaned_content = cleaned_content[1:-1]
        cleaned_content = ' '.join(i.strip() for i in cleaned_content)
        
        if cleaned_content[0] != '{' or cleaned_content[-1] != '}':
            print('🔻 Content not cleaned properly:')
            print(f'CONTENT:\n{content}')
            print(f'CLEANED CONTENT:\n{cleaned_content}')
        
        output = ast.literal_eval(cleaned_content)  # try parsing as JSON
        
        return output
    
    except SyntaxError as e:
        print(f'❌ SyntaxError: {e}')
        print(f'Content that caused the error:\n{cleaned_content}')
        traceback.print_exc()
        
        return None
    
    except Exception as e:
        print(f'❌ Unexpected error: {e}')
        print(f'Content that caused the error:\n{cleaned_content}')
        traceback.print_exc()
        
        return None
    

In [2]:
'''
COMMON FUNCTION WARNING: Align any changes to this function with the NER-Few-Shot version.
Function: set_download_path
Calls in: evaluate_distinct_entity, evaluate_all_entity
Description:
 - Determines the full file path for saving a file.
 - Takes a directory path and a file name as input.
 - If no directory path is provided, defaults to the current working directory.
 - Ensures the target directory exists by creating it if necessary.
 - Joins the directory path and file name to form the full file path.
 - Returns the complete download path.
'''

import os

def set_download_path(dir_path, file_name):
    if dir_path is None:
        dir_path = os.getcwd()

    os.makedirs(dir_path, exist_ok=True)
    download_path = os.path.join(dir_path, file_name)
    
    return download_path


In [ ]:
'''
COMMON FUNCTION WARNING: Align any changes to this function with the NER-Few-Shot version.
Function: evaluate_distinct_entity  
Calls in: Independent  
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Takes the labels, paragraphs, gold standard and predicted entities, and optional directory path as parameters.
 - Calculates precision, recall, and F1-score for each label within each paragraph.
 - Aggregates distinct entities across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging all paragraphs with their gold standard and predicted entities.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

import pandas as pd

# NEED PYTHON 3.9 OR MORE FOR TYPE DESCRIPTION
# def evaluate_distinct_entity(
#     label: list[str],
#     paragraph: list[str],
#     gold_entity: list[dict[str, list[str]]],
#     pred_entity: list[dict[str, list[str]]],
#     dir_path: str
# ) -> None:
    
def evaluate_distinct_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str=None) -> None:  
    # PARAGRAPH-LEVEL SCORE CALCULATION
    all_gold_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    all_pred_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieve paragraphs, gold standard entities, and predicted entities from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # log paragraphs with corresponding position, gold standard entities, and predicted entities
        file_name = 'paragraph-with-all-entity.txt'
        download_path = set_download_path(dir_path, file_name)
        
        with open(download_path, 'a', encoding='utf-8') as file:
            file.write(f'PARAGRAPH NUMBER: {para_index}\n')
            file.write(f'PARAGRAPH: {para}\n')
            file.write(f'GOLD ENTITY: {gold_ent}\n')
            file.write(f'PRED ENTITY: {pred_ent}\n')
            file.write('=======================================================\n')
        
        if dir_path:
            print(f'Downloaded {file_name} => {dir_path}')
        else:
            print(f'Downloaded {file_name} => current working directory')
        
        # retrieving labels from list
        for l in label: 
            # create entity set with unique entities
            unique_gold_ent = set(gold_ent[l])
            unique_pred_ent = set(pred_ent[l])
            
            # store entities (label-wise) for document-level calculation
            all_gold_ent[l].update(unique_gold_ent)
            all_pred_ent[l].update(unique_pred_ent)
            
            # calculate confusion matrix
            tp = unique_gold_ent & unique_pred_ent
            fp = unique_pred_ent - unique_gold_ent
            fn = unique_gold_ent - unique_pred_ent
            
            # calculate precision, recall and f1-score for each paragraph
            precision = len(tp) / (len(tp) + len(fp)) if unique_pred_ent else 0
            recall = len(tp) / (len(tp) + len(fn)) if unique_gold_ent else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # store mectrics for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # create dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para
    
    # download paragraph-level score in a spreadsheet
    file_name = 'score-para-DIS-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_para.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')
    
    # DOCUMENT-LEVEL SCORE CALCULATION
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # retrieve label from list
    for l in label:
        # calculate confusion matrix
        tp = all_gold_ent[l] & all_pred_ent[l]
        fp = all_pred_ent[l] - all_gold_ent[l]
        fn = all_gold_ent[l] - all_pred_ent[l]
        
        # calculate precision, recall and f1-score for entire document
        precision = len(tp) / (len(tp) + len(fp)) if all_pred_ent[l] else 0
        recall = len(tp) / (len(tp) + len(fn)) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # store mectrics for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])

    # create dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc

    # download document-level score in a spreadsheet
    file_name = 'score-doc-DIS-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_doc.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')
    

In [ ]:
'''
COMMON FUNCTION WARNING: Align any changes to this function with the NER-Few-Shot version.
Function: evaluate_all_entity  
Calls in: Independent  
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Takes the labels, paragraphs, gold standard and predicted entities, and optional directory path as parameters.
 - Calculates precision, recall, and F1-score for each label within each paragraph.
 - Aggregates all entities across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

import pandas as pd
from collections import Counter

def evaluate_all_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str=None) -> None:
    # PARAGRAPH-LEVEL SCORE CALCULATION
    all_gold_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    all_pred_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieve paragraphs, gold standard entities, and predicted entities from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # retrieve label from list
        for l in label:
            # count occurrence of each entity
            gold_counter = Counter(gold_ent[l])
            pred_counter = Counter(pred_ent[l])
            
            # store entities (label-wise) for document-level calculation
            all_gold_ent[l].update(gold_counter)  # CHECK: IF SAME TERM COMES FROM 2ND PARAGRAPH
            all_pred_ent[l].update(pred_counter)
            
            # calculate confusion matrix
            tp_counter = gold_counter & pred_counter  # Intersection of counts
            tp_sum = sum(tp_counter.values())

            fp_counter = pred_counter - gold_counter  # Predicted but not in gold
            fp_sum = sum(fp_counter.values())

            fn_counter = gold_counter - pred_counter  # Gold but not in predicted
            fn_sum = sum(fn_counter.values())

            # calculate precision, recall and f1-score for each paragraph
            precision = tp_sum / (tp_sum + fp_sum) if pred_counter else 0
            recall = tp_sum / (tp_sum + fn_sum) if gold_counter else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # store metrics for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # create dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para
    
    # download paragraph-level score in a spreadsheet
    file_name = 'score-para-ALL-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_para.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')

    # DOCUMENT-LEVEL SCORE CALCULATION
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # retrieve label from list
    for l in label:
        # calculate confusion matrix
        tp_counter = all_gold_ent[l] & all_pred_ent[l]
        tp_sum = sum(tp_counter.values())

        fp_counter = all_pred_ent[l] - all_gold_ent[l]
        fp_sum = sum(fp_counter.values())

        fn_counter = all_gold_ent[l] - all_pred_ent[l]
        fn_sum = sum(fn_counter.values())
        
        # calculate precision, recall and f1-score for entire document
        precision = tp_sum / (tp_sum + fp_sum) if all_pred_ent[l] else 0
        recall = tp_sum / (tp_sum + fn_sum) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # store metrics for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])
    
    # create dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc
    
    # download document-level score in a spreadsheet
    file_name = 'score-doc-ALL-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_doc.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')
       

In [ ]:
'''
COMMON FUNCTION WARNING: Align any changes to this function with the NER-Few-Shot version.
Function: process_input_data  
Calls in: Independent  
Description:
 - Reads evaluation data from a text file.
 - Separates the input into two lists: 
   - One for paragraphs.
   - One for corresponding entity annotation strings.
 - Parses the annotation strings into Python dictionaries using json module.
 - Returns two objects:
   - A list of paragraphs (str).
   - A list of corresponding entity annotations (dict).
'''

import json

def process_input_data(text_file):
    # read input data from text file
    with open(text_file, 'r', encoding='utf-8') as file:
        input_list = file.read().splitlines()

    # store paragraphs and annotations in different lists
    paragraph = []
    gold_ent_str = []

    for i in input_list:
        if input_list.index(i) == 0 or input_list.index(i) % 2 == 0:
            paragraph.append(i)
        else:
            gold_ent_str.append(i)

    # convert annotations (in string format) to nested object
    gold_entity = []

    for i in gold_ent_str:
        try:
            json_obj = json.loads(i)   # Convert to dictionary
            gold_entity.append(json_obj)  # Add to list of dictionaries
        except json.JSONDecodeError as e:
            print(f'Error decoding JSON for item: {i}\nError: {e}')

    return paragraph, gold_entity


In [12]:
'''
Function: annotate  
Calls in: Independent  
Description:
 - Executes ner_zero_shot function to perform NER using multiple instructions for each paragraph.
 - Takes instructions, examples, paragraphs, labels, gold standard entities, model name, temperature, and optional directory path as parameters.
 - For each paragraph:
   - Sends every instruction and corresponding examples to the LLM to extract predicted entities.
   - Collects and organizes the predicted entities under their respective labels.
   - Ensures all labels are present in the prediction output, inserting empty lists if needed.
   - Reorders predicted entities to match the original label order.
 - Saves variables -- label, paragraph, gold_entity, pred_entity -- in a Python file.
 - Returns a list of predicted entity dictionaries, aligned with each paragraph.
''' 

def annotate(instruction, example, paragraph, label, gold_entity, model, temp, dir_path):
    # predict entities from each paragraph for every instruction
    pred_entity = []
    for para in paragraph:
        pred_ent_in_para = dict()

        for inst, exmp in zip(instruction, example):
            response = ner_few_shot(inst, exmp, para, model, temp)
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                pred_ent_in_para[llm_label[0]] = llm_ent[0]

        # check for missing labels in predicted entities
        # all labels should be there even if they do not have any entities 
        if len(pred_ent_in_para) < len(label):
            print('Label missing in predicted data.')
            revised_data = dict()

            for l in label:
                if l not in pred_ent_in_para:
                    pred_ent_in_para[l] = []
                    print(f'Label -- {l} -- added to predicted data.')

            # organize the annotations' labels according to label's order
            for l in label:
                revised_data[l] = pred_ent_in_para[l]

            pred_ent_in_para = revised_data

        pred_entity.append(pred_ent_in_para)

    # save label, paragraph, gold_entity, pred_entity variables
    file_name = 'evaluation_variable.py'
    download_path = set_download_path(dir_path, file_name)
    
    with open(download_path, 'w', encoding='utf-8') as file:
        file.write('label = ' + repr(label) + '\n')
        file.write('paragraph = ' + repr(paragraph) + '\n')
        file.write('gold_entity = ' + repr(gold_entity) + '\n')
        file.write('pred_entity = ' + repr(pred_entity) + '\n')

    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')

    return pred_entity


In [13]:
from chatgpt_prompt import instruction, example_10

label = [
    'chemical',
    'material',
    'structure',
    'property',
    'application',
    'process',
    'equipment',
    'measurement',
    'abbreviation'
]

paragraph, gold_ent = process_input_data(text_file='sample-eval-data.txt')

pred_ent = annotate(
    instruction=instruction,
    example=example_10,
    paragraph=paragraph,
    label=label,
    gold_entity=gold_ent,
    model='gpt-4o',
    temp=0.2,
    dir_path='output/few-shot'
)

evaluate_distinct_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent,
    dir_path='output/few-shot'
)

evaluate_all_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent,
    dir_path='output/few-shot'
)


Label missing in predicted data.
Label -- equipment -- added to predicted data.
Label -- measurement -- added to predicted data.
Label -- abbreviation -- added to predicted data.
Label missing in predicted data.
Label -- equipment -- added to predicted data.
Label -- measurement -- added to predicted data.
Label -- abbreviation -- added to predicted data.
Label missing in predicted data.
Label -- equipment -- added to predicted data.
Label -- measurement -- added to predicted data.
Label -- abbreviation -- added to predicted data.
Downloaded evaluation_variable.py => output/few-shot
Downloaded paragraph-with-all-entity.txt => output/few-shot
Downloaded paragraph-with-all-entity.txt => output/few-shot
Downloaded paragraph-with-all-entity.txt => output/few-shot
Downloaded score-para-DIS-ENT.xlsx => output/few-shot
chemical:	0.43(f) | 0.32(p) | 0.67(r)
material:	0.35(f) | 0.29(p) | 0.44(r)
structure:	0.56(f) | 0.50(p) | 0.63(r)
property:	0.40(f) | 0.33(p) | 0.50(r)
application:	0.00(f) | 

# NEs Annotation Pipeline

In [3]:
'''
COMMON FUNCTION WARNING: Align any changes to this function with the NER-Few-Shot version.
Function: validate_entity_span  
Calls in: annotate (NEs Annotation Pipeline)
Description:
 - Validates predicted entity spans based on start and end indices within a paragraph.
 - Takes a list of predicted entities as parameter.
 - Converts the input into a DataFrame for structured validation:
   - A comparison column to check if the LLM-predicted term matches the sliced text.
   - An overlap flag to detect and remove overlapping entities.
 - Filters out entries where the predicted term does not match the sliced text or overlaps with another entity.
 - Returns a dictionary with a key 'entities', containing a list of validated entries.
'''

import pandas as pd

def validate_entity_span(entity_in_paragraph):
    columns = ['paragraph', 'start_index', 'end_index', 'label', 'llm_term', 'sliced_term']
    df = pd.DataFrame(entity_in_paragraph, columns=columns)
    df_sorted = df.sort_values(by='start_index')
    df_sorted.reset_index(drop=True, inplace=True)
    df_sorted['equal'] = df_sorted['llm_term'] == df_sorted['sliced_term']
    df_sorted['overlap'] = False

    for i in range(1, len(df_sorted)):
        if df_sorted.loc[i, 'start_index'] >= df_sorted.loc[i-1, 'start_index'] and \
            df_sorted.loc[i, 'start_index'] <= df_sorted.loc[i-1, 'end_index']:
            df_sorted.loc[i, 'overlap'] = True

    df_sorted = df_sorted[df_sorted['equal']]     # Keep only where equal is True
    df_sorted = df_sorted[~df_sorted['overlap']]  # Remove rows with overlap

    list_ = []
    for index, row in df_sorted.iterrows():
        list_.append([row['start_index'], row['end_index'], row['label']])

    validated_entity = {'entities': list_}

    return validated_entity


In [4]:
'''
Function: annotate  
Calls in: Independent  
Description:
 - Executes ner_zero_shot function to perform NER using multiple instructions for each paragraph.
 - Takes instructions, input text file, labels, model name, and temperature as parameters.
 - For each paragraph:
   - Sends every instruction to the LLM to extract predicted entities.
   - Collects and organizes the predicted entities under their respective labels.
   - Computes character-level start and end positions of each entity within the paragraph.
   - Creates a detailed record including paragraph index, label, entity string, and corresponding text slice.
   - Validates the entity spans using validate_entity_span function to remove incorrect or overlapping predictions.
 - Combines validated entities with their corresponding paragraph text.
 - Returns a dictionary with:
   - classes: A list of labels.
   - annotations: A list of [paragraph, validated_entities] pairs, formatted for spaCy-style annotation.
'''

def annotate(text_file, instruction, example, label, model, temp):
    # read evaluation data from text file and store paragraphs and entities in different list
    with open(text_file, 'r', encoding='utf-8') as file:
        paragraph = file.read().splitlines()

    # predict entities from each paragraph for every instruction
    para_with_ent = []
    for para in paragraph:
        para_index = paragraph.index(para)
        ent_in_para = dict()

        for inst, exmp in zip(instruction, example):
            response = ner_few_shot(inst, exmp, para, model, temp)
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                ent_in_para[llm_label[0]] = llm_ent[0]

        ent_detail_in_para = []

        for key in ent_in_para:
            end_index = 0

            for ent in ent_in_para[key]:
                ent_length = len(ent)
                start_index = para.find(ent, end_index)

                if start_index != -1:
                    end_index = start_index + ent_length
                    ent_detail = [
                        para_index,                  # Paragraph number within a file
                        start_index,                 # Starting position of an entity
                        end_index,                   # Ending position of an entity
                        key.upper(),                 # Label of an entity
                        ent,                         # Entity extracted by llm
                        para[start_index:end_index]  # Entity extracted using start and end position
                    ]

                    ent_detail_in_para.append(ent_detail)

        validated_ent = validate_entity_span(ent_detail_in_para)
        para_with_ent.append([para, validated_ent])

    spacy_annotation = {'classes': label, 'annotations': para_with_ent}
    
    return spacy_annotation


In [5]:
'''
Function: download_annotation  
Calls in: Independent  
Description:
 - Saves a Python dictionary object as a JSON file.
 - Takes a dictionary, and optional file name and directory path as parameters.
 - Calls set_download_path function to determine the full file path.
 - Writes the dictionary to the JSON file with an indentation of 4 for readability.
 - Downloads the JSON in custom location or current working directory.
'''

import json

def download_annotation(dict_obj, output_file_name='spacy_annotation', dir_path=None):
    file_name = f'{output_file_name}.json'
    download_path = set_download_path(dir_path, file_name)
    
    with open(download_path, 'w', encoding='utf-8') as json_file:
        json.dump(dict_obj, json_file, indent=4)
        
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => current working directory')
        

In [39]:
from chatgpt_prompt import instruction, example_20

label = [
    'CHEMICAL',
    'MATERIAL',
    'STRUCTURE',
    'PROPERTY',
    'APPLICATION',
    'PROCESS',
    'EQUIPMENT',
    'MEASUREMENT',
    'ABBREVIATION'
]

spacy_annotation = annotate(
    text_file='D:\\Drive\\SISE\\CelloGraph\\_dev\\data\\Wolf-2018.txt',
    instruction=instruction,
    example=example_20,
    label=label,
    model='gpt-4o',
    temp=0.2
)

download_annotation(
    dict_obj=spacy_annotation,
    output_file_name = 'Wolf-2018_spacy',
    dir_path='output/few-shot'
)
                    

❌ Unexpected error: string index out of range
Content that caused the error:
```python
{ "chemical": ["H <sub>2</sub> O", "O <sub>2</sub>", "CO <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <sub>2</sub> O", "O <sub>2</sub>", "H <s

Traceback (most recent call last):
  File "C:\Users\umayer\AppData\Local\Temp\ipykernel_6628\3105703422.py", line 40, in ner_few_shot
    if cleaned_content[0] != '{' or cleaned_content[-1] != '}':
IndexError: string index out of range


Downloaded Wolf-2018_spacy.json => output/few-shot
